In [1]:
import gymnasium as gym
import minigrid
from minigrid.wrappers import ImgObsWrapper
from ollama import chat
from ollama import ChatResponse
from anthropic import Anthropic
import os
import numpy as np
from dotenv import load_dotenv
import hashlib
import json

load_dotenv()

True

In [2]:
from typing import Literal
from pydantic import BaseModel

class Subgoal(BaseModel):
    action: Literal["pick_up", "open_door", "go_to_goal"]
    color: Literal["red", "green", "blue", "purple", "yellow", "grey"] | None = None
    object: Literal["key", "ball", "box"] | None = None

class SubgoalList(BaseModel):
    subgoals: list[Subgoal]

class CacheElement(BaseModel):
    hash: str
    mission:str
    environment: str
    subgoals: list[Subgoal]

In [3]:
class SubgoalCache:
    def __init__(self, path: str):
        self.path = path
        self._entries: dict[str, list[Subgoal]] = {}
        self._load()

    def _load(self) -> None:
        try:
            with open(self.path) as f:
                for line in f:
                    if line.strip():
                        rec = CacheElement.model_validate_json(line)
                        self._entries[rec.hash] = rec.subgoals
        except FileNotFoundError:
            pass

    def __contains__(self, key: str) -> bool:
        return key in self._entries

    def get(self, key: str) -> list[Subgoal] | None:
        return self._entries.get(key)

    def add(self, element: CacheElement) -> None:
        if element.hash in self._entries:
            return
        self._entries[element.hash] = element.subgoals
        with open(self.path, "a") as f:
            f.write(element.model_dump_json() + "\n")

    def __str__(self):
        l = []
        for k,v in self._entries.items():
            l.append(f"{k}, {v}")
        return "\n".join(l)

In [4]:
def make_prompt(mission: str, environment: str) -> str:
    return f"Mission: {mission}\nMap:\n{environment}"

def make_key(mission: str, environment: str) -> str:
    return hashlib.sha256(make_prompt(mission, environment).encode()).hexdigest()

In [5]:
OBJ   = {"wall":"W","floor":"F","door":"D","key":"K","ball":"A","box":"B","goal":"G","lava":"V"}
ADIR  = {0:">",1:"V",2:"<",3:"^"}
CCHAR = {"red":"R","green":"G","blue":"B","purple":"P","yellow":"Y","grey":"E"}

def render_map(base):
    g = base.grid
    rows = []
    for j in range(g.height):
        row = []
        for i in range(g.width):
            if (i, j) == tuple(base.agent_pos):
                row.append(ADIR[base.agent_dir] * 2)
            else:
                o = g.get(i, j)
                if o is None:
                    row.append("  ")
                elif o.type == "door":
                    row.append("__" if o.is_open else (("L" if o.is_locked else "D") + CCHAR[o.color]))
                else:
                    row.append(OBJ[o.type] + CCHAR[o.color])
        rows.append("".join(row))
    return "\n".join(rows)

In [6]:
SYSTEM_CORE = """You are a subgoal planner for an agent in a MiniGrid gridworld.
You are given the FULL map of the environment and the agent's mission.
Output the ordered list of subgoals the agent must complete to accomplish the mission.

HOW TO READ THE MAP
The map is printed row by row, from the top row down. Each cell is exactly 2 characters:
the first is the OBJECT, the second is its COLOR.

Objects (1st character):
   W = wall
   D = door
   K = key
   A = ball
   B = box
   G = goal
   V = lava
   F = floor
   '  ' (two spaces) = empty floor (walkable)

Colors (2nd character):
   R = red, G = green, B = blue, P = purple, Y = yellow, E = grey

So 'WE' = grey wall, 'GG' = green goal, 'VR' = lava (lava is always red),
'KY' = yellow key, 'AB' = blue ball.

Doors are special:
   L<color>  = a LOCKED door (e.g. 'LY' = locked yellow door)
   D<color>  = a closed but UNLOCKED door (e.g. 'DB' = closed blue door)
   __        = an already OPEN door (its color is not shown, and it needs no action)

The agent is two identical arrows showing the way it faces:
   >> east,  VV south,  << west,  ^^ north
Note: 'VV' (two arrows) is the agent facing south; lava is 'VR' (V plus a color).
The agent's cell shows the agent, not whatever it is standing on.

Coordinates: columns count left->right from 0 (i); rows count top->bottom from 0 (j);
a cell is (i, j). Use coordinates only to reason about what exists and what blocks what.

YOUR TASK
From the mission and the map, decide which subgoals the agent must achieve and in what order.
A LOCKED door ('L<color>') requires first picking up the matching-color key.
A closed unlocked door ('D<color>') only needs to be opened - no key.
An open door ('__') needs no subgoal at all.
To reach a room you must open the door(s) leading into it. Only propose subgoals for objects
that actually appear on the map or are named in the mission.

SUBGOAL FIELDS
Each subgoal has:
  action: one of 'pick_up', 'open_door', 'go_to_goal'
  color:  required for 'pick_up' and 'open_door' (red, green, blue, purple, yellow, grey)
  object: required for 'pick_up' (key, ball, box)
'go_to_goal' takes no color or object. List subgoals in the exact order the agent should perform them."""


# Anthropic / tool-use path: the schema carries the format, so no JSON talk at all
SYSTEM_TOOLS = SYSTEM_CORE + """

Report the plan by calling the emit_subgoals tool exactly once."""

# Ollama / format= path: this is where the JSON emission instructions belong
SYSTEM_OLLAMA = SYSTEM_CORE + """

YOUR OUTPUT
Return a JSON object with one key "subgoals": an ordered array of subgoal objects
with the fields described above. Output JSON only - no prose, no code fences."""



FEWSHOT = [
    # 1) No door, no key — go straight to the goal (teaches: don't invent keys/doors)
   (
      "Mission: get to the green goal square\nMap:\n" 
      "WEWEWEWEWEWE\n"
      "WE>>      WE\n"
      "WE        WE\n"
      "WE        WE\n"
      "WE      GGWE\n"
      "WEWEWEWEWEWE",
      {"subgoals": [
            {"action":"go_to_goal"}
      ]}
   ),

    # 2) Closed but UNLOCKED door (D, not L) — open it, no key needed.
    #    Also contrasts lava 'VR' with the south-facing agent 'VV'.
    ("Mission: open the door and then get to the goal\nMap:\n"
     "WEWEWEWEWEWE\n"
     "WEVV  WE  WE\n"
     "WE    DR  WE\n"
     "WEVR  WE  WE\n"
     "WE    WEGGWE\n"
     "WEWEWEWEWEWE",
     '{"subgoals":[{"action":"open_door","color":"red"},{"action":"go_to_goal"}]}'),

    # 3) DoorKey 6x6, blue — locked door needs the matching key first (color/size variant)
    ("Mission: use the key to open the door and then get to the goal\nMap:\n"
     "WEWEWEWEWEWE\n"
     "WEVV  WE  WE\n"
     "WEKB  LB  WE\n"
     "WE    WE  WE\n"
     "WE    WEGGWE\n"
     "WEWEWEWEWEWE",
     '{"subgoals":[{"action":"pick_up","color":"blue","object":"key"},{"action":"open_door","color":"blue"},{"action":"go_to_goal"}]}'),

    # 4) MultiRoom — a chain of closed unlocked doors, opened in traversal order.
    #    'DE' is a GREY door (E = grey), which is easy to misread as an object letter.
    ("Mission: traverse the rooms to get to the goal\nMap:\n"
     "WEWEWEWEWEWEWEWEWE  \n"
     "WE      WE      WE  \n"
     "WE>>    DP      WE  \n"
     "WE      WE      WE  \n"
     "WEWEWEWEWEWEDEWEWEWE\n"
     "          WE      WE\n"
     "          WE      WE\n"
     "          WE    GGWE\n"
     "          WEWEWEWEWE",
     '{"subgoals":[{"action":"open_door","color":"purple"},{"action":"open_door","color":"grey"},{"action":"go_to_goal"}]}'),

    # 5) DoorKey 8x8, yellow — placed last so it's adjacent to the live query
    ("Mission: use the key to open the door and then get to the goal\nMap:\n"
     "WEWEWEWEWEWEWEWE\n"
     "WE>>  WE      WE\n"
     "WE    WE      WE\n"
     "WE  KYWE      WE\n"
     "WE    LY      WE\n"
     "WE    WE      WE\n"
     "WE    WE    GGWE\n"
     "WEWEWEWEWEWEWEWE",
     '{"subgoals":[{"action":"pick_up","color":"yellow","object":"key"},{"action":"open_door","color":"yellow"},{"action":"go_to_goal"}]}'),
]

In [7]:
env = gym.make("MiniGrid-MultiRoom-N4-S5-v0", render_mode="human")
base = env.unwrapped
base.reset(seed=42)

messages = [{"role": "system", "content": SYSTEM_OLLAMA}]
for u, a in FEWSHOT:
    messages += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
messages.append({"role": "user", "content": f"Mission: {base.mission}\nMap:\n{render_map(base)}"})

# resp = chat(
#     model="qwen3.5:4b", #"qwen3.5:4b",
#     messages=messages, 
#     options={"temperature": 0}, 
#     think=False,
#     format={
#         "type": "object",
#         "properties": {
#           "subgoals": {
#             "type": "array",
#             "items": {
#               "type": "object",
#               "properties": {
#                 "action": { "type": "string", "enum": ["pick_up", "open_door", "go_to_goal"] },
#                 "color":  { "type": "string", "enum": ["red","green","blue","purple","yellow","grey"] },
#                 "object": { "type": "string", "enum": ["key","ball","box"] }
#             },
#             "required": ["action"]
#             }
#         }
#         },
#         "required": ["subgoals"]
#     }
# )

# resp.message.content

/Users/oleroessler/Documents/Universitaet/Masters Degree/04 Projects/rl_final/.venv/lib/python3.13/site-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment MiniGrid-MultiRoom-N4-S5-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(
/Users/oleroessler/Documents/Universitaet/Masters Degree/04 Projects/rl_final/.venv/lib/python3.13/site-packages/pygame/sysfont.py:494: UserWarning: The system font 'freesansbold.ttf' couldn't be found. Did you mean: 'freesansbold', 'freesans'? Verify your font name input. Using the default font instead.
  warnings.warn(


In [8]:
def fetch_subgoals(client, prompt: str, model: str="claude-haiku-4-5") -> list[Subgoal]:
    resp = client.messages.create(
        model=model,
        max_tokens=1024,
        system=SYSTEM_TOOLS,
        tools=[{
            "name": "emit_subgoals",
            "description": "Return the ordered subgoals the agent must achieve.",
            "input_schema": SubgoalList.model_json_schema(),
        }],
        tool_choice={"type": "tool", "name": "emit_subgoals"},
        messages=[{"role": "user", "content": prompt}],
    )
    raw = next(b.input for b in resp.content if b.type == "tool_use")
    return SubgoalList.model_validate(raw).subgoals

In [ ]:
cache = SubgoalCache("subgoal_cache.jsonl")
client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))


env = gym.make("MiniGrid-MultiRoom-N4-S5-v0", render_mode="human")
base = env.unwrapped
base.reset(seed=42)

# make key for current env 
key = make_key(base.mission, render_map(base))
prompt = make_prompt(base.mission, render_map(base))

if key not in cache:
    cache.add(
        CacheElement(hash=key, 
                     mission=base.mission, 
                     environment=render_map(base), 
                     subgoals=fetch_subgoals(client, prompt)
        )
    )

subgoals = cache.get(key)

print(subgoals)

[Subgoal(action='open_door', color='blue', object=None), Subgoal(action='go_to_goal', color=None, object=None)]


: 